# Import Libraries

Import the Python libraries required for data processing, analysis, and visualization throughout the project.

In [1]:
import pandas as pd
import zipfile

#Load Datasets


Load the S&P 500 Sustainability Reports dataset and the ESG Risk Ratings dataset into pandas dataframes for further analysis.

In [4]:
# Extract the sustainability reports dataset
with zipfile.ZipFile("preprocessed_content.csv.zip", "r") as zip_ref:
    zip_ref.extractall("reports_data")

# Load the extracted CSV
reports = pd.read_csv("reports_data/preprocessed_content.csv")

# Load ESG dataset
esg = pd.read_csv("SP 500 ESG Risk Ratings.csv")

print("=== REPORTS DATASET ===")
print("Shape:", reports.shape)

print("\nColumns:")
print(reports.columns.tolist())

print("\nFirst 5 Rows:")
print(reports.head())

print("\n=========================\n")

print("=== ESG DATASET ===")
print("Shape:", esg.shape)

print("\nColumns:")
print(esg.columns.tolist())

print("\nFirst 5 Rows:")
print(esg.head())

=== REPORTS DATASET ===
Shape: (866, 10)

Columns:
['Unnamed: 0', 'filename', 'ticker', 'year', 'preprocessed_content', 'ner_entities', 'e_score', 's_score', 'g_score', 'total_score']

First 5 Rows:
   Unnamed: 0          filename ticker  year  \
0           0  ASX_BSX_2020.pdf    BSX  2020   
1           1  ASX_BSX_2022.pdf    BSX  2022   
2           2  ASX_EXR_2022.pdf    EXR  2022   
3           3  LSE_ADM_2019.pdf    ADM  2019   
4           4  LSE_ADM_2020.pdf    ADM  2020   

                                preprocessed_content  \
0  style guide colour colour use imagecolour prof...   
1  sustainability report look mining green office...   
2  report environment social governance esg basel...   
3  corporate social responsibilty report introduc...   
4  sustainability admiral commit maintain respons...   

                                        ner_entities  e_score  s_score  \
0  ['bk%', 'rgb', 'un', 'el ectric mine consortiu...     3.16    18.00   
1  ['murray street', 'west 

# Merge Datasets

Combine the sustainability reports dataset with the ESG risk ratings dataset using company ticker symbols to create a unified dataset.

In [5]:
print("Unique tickers in reports:", reports['ticker'].nunique())

print("Unique symbols in ESG:", esg['Symbol'].nunique())

# Find overlap
common = set(reports['ticker']) & set(esg['Symbol'])

print("\nCompanies present in BOTH datasets:")
print(len(common))

print("\nFirst 20 matching companies:")
print(list(common)[:20])

Unique tickers in reports: 263
Unique symbols in ESG: 503

Companies present in BOTH datasets:
262

First 20 matching companies:
['LKQ', 'USB', 'ADM', 'TMUS', 'CHRW', 'PANW', 'ADP', 'CCL', 'NTAP', 'CFG', 'SCHW', 'PKG', 'DLTR', 'LH', 'JNPR', 'FCX', 'BXP', 'EIX', 'CNP', 'CPB']


In [6]:
# Merge datasets

merged = reports.merge(
    esg,
    left_on="ticker",
    right_on="Symbol",
    how="inner"
)

print("Merged Dataset Shape:")
print(merged.shape)

print("\nColumns:")
print(merged.columns.tolist())

print("\nFirst 5 Rows:")
print(merged.head())

Merged Dataset Shape:
(862, 25)

Columns:
['Unnamed: 0', 'filename', 'ticker', 'year', 'preprocessed_content', 'ner_entities', 'e_score', 's_score', 'g_score', 'total_score', 'Symbol', 'Name', 'Address', 'Sector', 'Industry', 'Full Time Employees', 'Description', 'Total ESG Risk score', 'Environment Risk Score', 'Governance Risk Score', 'Social Risk Score', 'Controversy Level', 'Controversy Score', 'ESG Risk Percentile', 'ESG Risk Level']

First 5 Rows:
   Unnamed: 0          filename ticker  year  \
0           0  ASX_BSX_2020.pdf    BSX  2020   
1           1  ASX_BSX_2022.pdf    BSX  2022   
2           2  ASX_EXR_2022.pdf    EXR  2022   
3           3  LSE_ADM_2019.pdf    ADM  2019   
4           4  LSE_ADM_2020.pdf    ADM  2020   

                                preprocessed_content  \
0  style guide colour colour use imagecolour prof...   
1  sustainability report look mining green office...   
2  report environment social governance esg basel...   
3  corporate social responsib

In [7]:
print("Merged Dataset Shape:", merged.shape)

print("\nMissing Values:")
print(merged.isnull().sum().sort_values(ascending=False).head(15))

print("\nUnique Companies:")
print(merged['ticker'].nunique())

print("\nUnique Industries:")
print(merged['Industry'].nunique())

print("\nUnique Sectors:")
print(merged['Sector'].nunique())

print("\nYears Covered:")
print(sorted(merged['year'].unique()))

Merged Dataset Shape: (862, 25)

Missing Values:
Controversy Score         49
Full Time Employees       11
Total ESG Risk score       1
Social Risk Score          1
Environment Risk Score     1
ESG Risk Percentile        1
Controversy Level          1
Governance Risk Score      1
ESG Risk Level             1
g_score                    0
s_score                    0
e_score                    0
ner_entities               0
preprocessed_content       0
year                       0
dtype: int64

Unique Companies:
262

Unique Industries:
93

Unique Sectors:
11

Years Covered:
[np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


# Text Cleaning

The sustainability reports dataset (preprocessed_content.csv)
already contains cleaned and preprocessed text. Therefore,
additional text cleaning steps were not required.

# Generate Features

Generate features by counting concrete, vague, and unverifiable keywords in the sustainability reports. These features are used to assign claim types and support subsequent research question analyses.

In [8]:
concrete_keywords = [
    "percent", "reduction", "target", "achieve",
    "tons", "kg", "mw", "co2", "emissions"
]

vague_keywords = [
    "commitment", "future", "vision",
    "leadership", "sustainability",
    "ambition", "responsible"
]

unverifiable_keywords = [
    "best", "leading",
    "world-class",
    "most sustainable",
    "industry-leading"
]

print("Concrete:", concrete_keywords)
print("Vague:", vague_keywords)
print("Unverifiable:", unverifiable_keywords)

Concrete: ['percent', 'reduction', 'target', 'achieve', 'tons', 'kg', 'mw', 'co2', 'emissions']
Vague: ['commitment', 'future', 'vision', 'leadership', 'sustainability', 'ambition', 'responsible']
Unverifiable: ['best', 'leading', 'world-class', 'most sustainable', 'industry-leading']


In [9]:
def count_keywords(text, keywords):
    text = str(text).lower()
    count = 0

    for word in keywords:
        count += text.count(word)

    return count


merged['concrete_count'] = merged['preprocessed_content'].apply(
    lambda x: count_keywords(x, concrete_keywords)
)

merged['vague_count'] = merged['preprocessed_content'].apply(
    lambda x: count_keywords(x, vague_keywords)
)

merged['unverifiable_count'] = merged['preprocessed_content'].apply(
    lambda x: count_keywords(x, unverifiable_keywords)
)

print(
    merged[['ticker',
            'concrete_count',
            'vague_count',
            'unverifiable_count']].head()
)

  ticker  concrete_count  vague_count  unverifiable_count
0    BSX              22          101                   0
1    BSX              28          120                   0
2    EXR              12            4                   0
3    ADM              18           29                   0
4    ADM              24           72                   0


In [10]:
def assign_label(row):

    if row['concrete_count'] >= row['vague_count'] and \
       row['concrete_count'] >= row['unverifiable_count']:
        return "Concrete"

    elif row['vague_count'] >= row['concrete_count'] and \
         row['vague_count'] >= row['unverifiable_count']:
        return "Vague"

    else:
        return "Unverifiable"


merged['claim_type'] = merged.apply(assign_label, axis=1)

print(merged['claim_type'].value_counts())

claim_type
Vague       753
Concrete    109
Name: count, dtype: int64


# Save Master Dataset

Store the final processed dataset containing all generated features and claim labels. This master dataset will be used in subsequent notebooks for answering the research questions.

In [11]:
# Save the final processed dataset
merged.to_csv("Master_Dataset.csv", index=False)

print("Master Dataset saved successfully.")
print("Shape:", merged.shape)

Master Dataset saved successfully.
Shape: (862, 29)
